# Simple XRK Data Analysis with Libxrk

This notebook demonstrates using libxrk to do easy analysis of AIM(tm) datasets without needing any AIM software installed locally.

**Note:** This notebook works in both JupyterLite (browser) and standard JupyterLab environments.

In [ ]:
# Install required packages (needed for JupyterLite, skipped in regular JupyterLab if already installed)
%pip install -q pandas plotly libxrk motorsports-data-notebook jinja2

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import core libraries
import pandas as pd

# Visualization libraries
import plotly.express as px
import plotly.graph_objects as go

# Import libxrk:
from libxrk import aim_xrk

# Import helper functions (includes show_fig for JupyterLite compatibility)
from motorsports_data_notebook import show_fig, get_best_lap, compute_start_line, plot_lap_gps

In [ ]:
log = aim_xrk("CMD_Inferno 86_Fuji GP Sh_Generic testing_a_2248.xrz")

division: scan=0.077222, gps=0.010938, group/ch=0.021427 more


/home/m3rlin45/.cache/pypoetry/virtualenvs/motorsports-data-notebook-qUYIIvW5-py3.12/lib/python3.12/site-packages/libxrk/gps.py:334: RuntimeWarning: invalid value encountered in divide
  t = np.maximum(-np.sum(SN * O, axis=1) / np.sum(SN * D, axis=1), 0)
/home/m3rlin45/.cache/pypoetry/virtualenvs/motorsports-data-notebook-qUYIIvW5-py3.12/lib/python3.12/site-packages/libxrk/gps.py:347: RuntimeWarning: divide by zero encountered in divide
  t = np.maximum(-np.sum(SN * O, axis=1) / np.sum(SN * D, axis=1), 0)
/home/m3rlin45/.cache/pypoetry/virtualenvs/motorsports-data-notebook-qUYIIvW5-py3.12/lib/python3.12/site-packages/libxrk/gps.py:348: RuntimeWarning: invalid value encountered in multiply
  dist = np.sum(np.square(O + t.reshape((len(t), 1)) * D), axis=1)


In [4]:
# Flatten all the data into a uniform table, interpolating as needed (can use a lot of RAM!)
channels = log.get_channels_as_table().to_pandas()


In [5]:

laps = log.laps.to_pandas()
laps['lap_time'] = pd.to_timedelta(laps['end_time'] - laps['start_time'], unit='ms')

laps.style.format({'lap_time': lambda x: f"{int(x.total_seconds() // 60)}:{x.total_seconds() % 60:06.3f}"})

,num,start_time,end_time,lap_time
0,0,0,150454,2:30.454
1,1,150454,279602,2:09.148
2,2,279602,406240,2:06.638
3,3,406240,532797,2:06.557
4,4,532797,659282,2:06.485
5,5,659282,787773,2:08.491
6,6,787773,913776,2:06.003
7,7,913776,1041397,2:07.621
8,8,1041397,1168322,2:06.925
9,9,1168322,1294676,2:06.354


In [15]:
channels.columns

Index(['timecodes', 'AmbientTemp', 'Baro', 'Best Run Diff', 'Best Today Diff',
       'BrakePress', 'BrakeSw', 'CAT1', 'CH', 'ClutchSw', 'ECT',
       'External Voltage', 'FL_Ch1', 'FL_Ch2', 'FL_Ch3', 'FL_Ch4', 'FL_Ch5',
       'FL_Ch6', 'FL_Ch7', 'FL_Ch8', 'FR_Ch1', 'FR_Ch2', 'FR_Ch3', 'FR_Ch4',
       'FR_Ch5', 'FR_Ch6', 'FR_Ch7', 'FR_Ch8', 'GPS Altitude', 'GPS Latitude',
       'GPS Longitude', 'GPS Speed', 'Gear', 'InlineAcc', 'IntakeAirT',
       'LF_Shock_Pot', 'LR_Shock_Pot', 'Lambda', 'LateralAcc', 'LoggerTemp',
       'Luminosity', 'MAP', 'OilTemp', 'PPS', 'PitchRate', 'Predictive Time',
       'Prev Lap Diff', 'RF_Shock_Pot', 'RL_Ch1', 'RL_Ch2', 'RL_Ch3', 'RL_Ch4',
       'RL_Ch5', 'RL_Ch6', 'RL_Ch7', 'RL_Ch8', 'RPM', 'RR_Ch1', 'RR_Ch2',
       'RR_Ch3', 'RR_Ch4', 'RR_Ch5', 'RR_Ch6', 'RR_Ch7', 'RR_Ch8',
       'RR_Shock_Pot', 'Ref Lap Diff', 'RollRate', 'SpeedAverage',
       'SteerAngle', 'TPMS_ALM_LF', 'TPMS_ALM_LR', 'TPMS_ALM_RF',
       'TPMS_ALM_RR', 'TPMS_Press_LF', 'TP

In [6]:
# Best lap extraction and interactive Plotly GPS speed plot (km/h)
# Get best lap
best_lap = get_best_lap(laps)
start_ts = best_lap['start_time']
end_ts = best_lap['end_time']
channels['speed_kmh'] = channels['GPS Speed'] * 3.6
lap_channels = channels.query(f'timecodes >= @start_ts and timecodes <= @end_ts').copy()

In [ ]:

# plot speed on GPS map
fig = plot_lap_gps(
    lat=lap_channels['GPS Latitude'],
    lon=lap_channels['GPS Longitude'],
    color_channels=[
        (lap_channels['speed_kmh'], 'Speed (km/h)', 'Viridis')
    ],
    title='Speed'
)
show_fig(fig)

In [ ]:

# Plot with multiple color channels
fig = plot_lap_gps(
    lat=lap_channels['GPS Latitude'],
    lon=lap_channels['GPS Longitude'],
    color_channels=[
        (lap_channels['BrakePress'], 'BrakePres', 'Reds'),
        (lap_channels['PPS'], 'Throttle', 'Greens')
    ],
    title='Accelerator and Brake Pressure on Best Lap'
)
show_fig(fig)

In [ ]:
# Front Tire Thermography - Best Lap
# Extract FL and FR tire temperature channels
# Ch1 = leftmost (outside for FL, inside for FR), Ch8 = rightmost (inside for FL, outside for FR)
fl_channels = ['FL_Ch1', 'FL_Ch2', 'FL_Ch3', 'FL_Ch4', 'FL_Ch5', 'FL_Ch6', 'FL_Ch7', 'FL_Ch8']
fr_channels = ['FR_Ch1', 'FR_Ch2', 'FR_Ch3', 'FR_Ch4', 'FR_Ch5', 'FR_Ch6', 'FR_Ch7', 'FR_Ch8']

fl_temps = lap_channels[fl_channels].values.T  # Shape: (8 channels, n_samples)
fr_temps = lap_channels[fr_channels].values.T

# Calculate distance traveled using GPS Speed (m/s) integrated over time
time_s = (lap_channels['timecodes'] - lap_channels['timecodes'].iloc[0]) / 1000.0
speed_ms = lap_channels['GPS Speed']
dt = time_s.diff().fillna(0)
distance_m = (speed_ms * dt).cumsum()

# Get color scale range across both tires for consistent coloring
vmin = min(fl_temps.min(), fr_temps.min())
vmax = max(fl_temps.max(), fr_temps.max())

# Create subplots with shared x-axis
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=('Front Left Tire (Outside at top)', 'Front Right Tire (Outside at bottom)')
)

# Front Left heatmap - FL Ch1 (outside/left) at top
fig.add_trace(
    go.Heatmap(
        z=fl_temps,
        x=distance_m.values,
        y=fl_channels,
        colorscale='Inferno',
        zmin=vmin, zmax=vmax,
        showscale=False
    ),
    row=1, col=1
)
fig.update_yaxes(autorange='reversed', row=1, col=1)  # Ch1 at top

# Front Right heatmap - FR Ch8 (outside/right) at bottom
fig.add_trace(
    go.Heatmap(
        z=fr_temps,
        x=distance_m.values,
        y=fr_channels,
        colorscale='Inferno',
        zmin=vmin, zmax=vmax,
        colorbar=dict(title='Temp (°C)')
    ),
    row=2, col=1
)
fig.update_yaxes(autorange='reversed', row=2, col=1)  # Ch1 at top, Ch8 at bottom

fig.update_layout(
    title='Front Tire Temperatures - Best Lap',
    xaxis2_title='Distance (m)',
    yaxis_title='FL',
    yaxis2_title='FR',
    width=900,
    height=500
)

show_fig(fig)